# Choose Fisher accuracy using one default-accuracy calculation

This notebook follows the numbered Cobaya examples and calls the shared
CosmoLike Fisher functions directly, as in LSST Y1. The expensive starting
calculation differentiates every listed parameter once at each of two step
sizes. Its parameter correlations identify which derivative columns matter
most for the marginalized $(w_0,w_a)$ figure of merit. Subsequent screens
recompute only those columns. A final full calculation checks the settings
selected by that screen.

Choose NLA or TATT with the numbered YAML below. The fixed simulation covariance,
source reconstruction, IA normalization and supplied bispectrum remain those of
the YAML. Opening dark energy is an explicit change for this diagnostic.
**These cells have not been rerun during the refactor.** Running them evaluates
real theory models; no saved historical forecast is substituted automatically.

These examples use the unmasked, uncompressed row order of numbered examples 1
and 2. A different dataset mask or compression requires matching row labels
before using the two-/three-point split shown here.


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import os
# BLAS must not start another worker team inside the native OpenMP calculation.
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "4"

from copy import deepcopy
from pathlib import Path
import sys
import importlib
import numpy as np
import matplotlib
from matplotlib import pyplot as plt
from scipy.linalg import solve_triangular
from cobaya.model import get_model
from cobaya.yaml import yaml_load_file, yaml_dump_file

project = Path.cwd().resolve()
while not (project / "EXAMPLE_EVALUATE3.yaml").exists():
    if project == project.parent:
        raise FileNotFoundError("Open this notebook from the DES project directory.")
    project = project.parent
cocoa = project.parents[1]
sys.path.insert(0, str(project / "interface"))
sys.path.insert(0, str(cocoa / "external_modules/code/cosmolike_core"))
import cosmolike_des_3pcf_interface as ci
import cosmolike_notebook_utils.fisher as cnu

# Numbered Cocoa YAMLs resolve package and dataset paths from the Cocoa directory.
os.chdir(cocoa)

In [ ]:
# Figure style of the whole notebook: the shared plotting functions never change
# rcParams, so these lines set every font, tick and grid default.
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.family'] = 'STIXGeneral'
matplotlib.rcParams['mathtext.rm'] = 'Bitstream Vera Sans'
matplotlib.rcParams['mathtext.it'] = 'Bitstream Vera Sans:italic'
matplotlib.rcParams['mathtext.bf'] = 'Bitstream Vera Sans:bold'
matplotlib.rcParams['xtick.bottom'] = True
matplotlib.rcParams['xtick.top'] = False
matplotlib.rcParams['ytick.right'] = False
matplotlib.rcParams['axes.edgecolor'] = 'black'
matplotlib.rcParams['axes.linewidth'] = '1.0'
matplotlib.rcParams['axes.labelsize'] = 'medium'
matplotlib.rcParams['axes.grid'] = True
matplotlib.rcParams['grid.linewidth'] = '0.0'
matplotlib.rcParams['grid.alpha'] = '0.18'
matplotlib.rcParams['grid.color'] = 'lightgray'
matplotlib.rcParams['legend.labelspacing'] = 0.77
matplotlib.rcParams['savefig.bbox'] = 'tight'
matplotlib.rcParams['savefig.format'] = 'pdf'

## 1. State the parameter space and its physical steps

In [ ]:
# Example 1 is NLA; example 2 is TATT. Run this notebook separately for each.
example = "EXAMPLE_EVALUATE4.yaml"
info = yaml_load_file(str(project / example))
fiducial = dict(info["sampler"]["evaluate"]["override"])
info.pop("sampler")
info.pop("output", None)
info["likelihood"]["des_3pcf.joint_shear"]["omp_threads"] = 4

# Unlike the numbered inference example, this diagnostic varies dark energy.
# Removing the fixed values lets Cobaya pass the two stencil coordinates to CAMB.
info["params"]["w"] = {
    "prior": {"min": -2.0, "max": -0.3},
    "ref": -1.0,
}
info["params"]["wa"] = {
    "prior": {"min": -2.0, "max": 2.0},
    "ref": 0.0,
}
fiducial["w"] = -1.0
fiducial["wa"] = 0.0

# These are absolute physical displacements, not percentages of a fiducial.
# Small nuisance steps and larger dark-energy steps therefore have explicit units.
steps = {
    "omegam": 0.003,
    "As_1e9": 0.024,
    "omegab": 0.0005,
    "H0": 0.5,
    "ns": 0.005,
    "DES_A1_1": 0.01,
    "DES_A1_2": 0.03,
    "DES_M1": 0.001,
    "DES_M2": 0.001,
    "DES_M3": 0.001,
    "DES_M4": 0.001,
    "DES_DZ_S1": 0.001,
    "DES_DZ_S2": 0.001,
    "DES_DZ_S3": 0.001,
    "DES_DZ_S4": 0.001,
    "w": 0.02,
    "wa": 0.04,
}
# Gaussian source-calibration widths enter once. Flat prior limits do not
# supply Gaussian information; their posterior truncation is absent here.
priors = {
    "DES_M1": 0.0091,
    "DES_M2": 0.0078,
    "DES_M3": 0.0076,
    "DES_M4": 0.0076,
    "DES_DZ_S1": 0.018,
    "DES_DZ_S2": 0.015,
    "DES_DZ_S3": 0.011,
    "DES_DZ_S4": 0.017,
}

if example == "EXAMPLE_EVALUATE4.yaml":
    steps["DES_A2_1"] = 0.02
    steps["DES_A2_2"] = 0.03
    steps["DES_BTA_1"] = 0.02

In [ ]:
# CAMB's derived sigma8 at every evaluated point, keyed by the point.
# As_1e9 is sampled and sigma8 is derived, so S8 = sigma8 (Omega_m/0.3)^(1/2)
# depends on every cosmological parameter. get_s8 differentiates it with the
# same stencils as the data vector, so it needs no CAMB evaluation of its own.
sigma8_values = {}


def get_dv(parameters):
    """Evaluate the current numbered Cobaya model at a physical parameter point.

    Cobaya runs its declared theory blocks before the DES likelihood. The
    likelihood retains the full mean it evaluated; the native selector applies
    its data mask and compression. A Fisher displacement must reach that same
    prediction, not another CAMB or bispectrum preparation route.

    Arguments:
      parameters = dictionary of sampled physical parameters for the current model.
    Returns:
      Owned one-dimensional selected data vector, in the covariance's row order.
    Raises:
      RuntimeError if the likelihood evaluation is not finite.
    Side effects:
      Updates the current model and the process-global native cosmology,
      and records CAMB's derived sigma8 at this point in sigma8_values.
    """
    loglikes, derived = model.loglikes(
        params_values=parameters,
        return_derived=True,
        as_dict=True,
        cached=False,
    )
    if not np.all(np.isfinite(list(loglikes.values()))):
        raise RuntimeError("A Fisher point has nonfinite likelihood; inspect that point.")
    sigma8_values[tuple(sorted(parameters.items()))] = derived["sigma8"]
    return np.array(
        object=ci.select_joint_data_vector(prediction=likelihood.last_datavector),
        dtype=np.float64,
        copy=True,
    )


def get_s8(parameters):
    """Return S8 = sigma8 (Omega_m/0.3)^(1/2) at a point, as a one-entry vector.

    sigma8 is CAMB's derived value that get_dv recorded at the same point.
    Passing get_s8 to cnu.differentiate with the data-vector steps gives the
    gradient dS8/dp of every parameter; the Omega_m entry includes both the
    explicit square root and the change of sigma8.

    Arguments:
      parameters = dictionary of sampled physical parameters, with omegam.
    Returns:
      One-dimensional array holding S8.
    Side effects:
      Evaluates the model through get_dv only if the point is not recorded.
    """
    key = tuple(sorted(parameters.items()))
    if key not in sigma8_values:
        get_dv(parameters=parameters)
    return np.array([sigma8_values[key] * np.sqrt(parameters["omegam"] / 0.3)])

## 2. Compute the full base at default accuracy

The derivative columns are physical derivatives, with one column for each
name in `steps`. The five-point rules at $h$ and $h/2$ expose step sensitivity.
Numerical integrations can still share a smooth error at both steps, so this
agreement alone does not prove convergence.

The shared mean cache retains identical physical points **inside this single
fixed-accuracy calculation**. It avoids repeating the common center and shared
stencil nodes. A new model and a new cache are constructed for every accuracy
case; a vector at one sampling cannot serve as a vector at another sampling.

In [ ]:
model = get_model(info_or_yaml_or_file=info)
likelihood = model.likelihood["des_3pcf.joint_shear"]
sigma8_values.clear()
try:
    covariance = np.array(object=ci.get_joint_covariance(), copy=True)
    row_kind = likelihood.rows.kind.copy()
    predict = cnu.ExactMeanCache(predict=get_dv)
    names, center, dh, dh2 = cnu.differentiate(
        predict=predict,
        parameters=fiducial,
        steps=steps,
    )
    base = cnu.fisher_matrix(
        derivatives=dh2,
        names=names,
        covariance=covariance,
        priors=priors,
    )
    base_h = cnu.fisher_matrix(
        derivatives=dh,
        names=names,
        covariance=covariance,
        priors=priors,
    )
    # dS8/dp from CAMB's sigma8 at the stencil points just evaluated.
    # The accuracy cases below change only likelihood settings, never
    # CAMB, so this one gradient serves every case.
    s8_names, _, _, s8_h2 = cnu.differentiate(
        predict=get_s8,
        parameters=fiducial,
        steps=steps,
    )
    assert s8_names == names
    s8_gradient = s8_h2[0]
    iw0 = names.index("w")
    iwa = names.index("wa")
    invcov = np.linalg.solve(covariance, np.eye(covariance.shape[0]))
    eps = np.asarray(list(steps.values()), dtype=float) / 2.0
    weights = cnu.get_fom_noise_weights(F=base, i=iw0, j=iwa, eps=eps)
    # Absolute offsets avoid rescaling the stencil by a signed or zero fiducial.
    offset_dv = cnu.make_offset_predictor(
        predict=predict,
        reference=fiducial,
        names=names,
    )
    baseline_a = np.zeros(len(names))
    baseline_sigma = np.zeros(len(names))
    for index in range(len(names)):
        noise = cnu.get_stencil_noise(
            dv=offset_dv,
            index=index,
            h=eps[index],
            CV=np.zeros(len(names)),
            invcov=invcov,
        )
        baseline_a[index] = noise["a"]
        baseline_sigma[index] = noise["sigma"]
    print("FoM at h:", cnu.get_FoM(i=iw0, j=iwa, F=base_h))
    print("FoM at h/2:", cnu.get_FoM(i=iw0, j=iwa, F=base))
finally:
    model.close()

## 3. Estimate the positive information from derivative noise

Five equally spaced predictions form a fourth difference:

$$\Delta^4m=m(-2h)-4m(-h)+6m(0)-4m(h)+m(2h).$$

Its squared covariance-weighted norm divided by 70 estimates the evaluation
noise variance under independent, equal-variance numerical fluctuations. The
factor 70 is the sum of the squared coefficients. This estimate supplies the
false derivative information $a_j=(130/144)\hat\sigma_j^2/h_j^2$.

The full base Fisher matrix determines how that column affects the dark-energy
ellipse after marginalizing all listed parameters. The shared weights give
$\delta\ln\mathrm{FoM}\simeq\tfrac12\sum_jR_j^2\sigma_j^2a_j$.
The next cell selects columns by their **measured baseline contribution**, then
prints the omitted contribution. A small baseline contribution is not a bound
on its value at another accuracy. Smooth fourth derivatives and correlated
errors also violate the independent-noise interpretation.

In [ ]:
# R2 times sigma2 carries parameter units squared; multiplying by a
# produces a dimensionless contribution to the logarithmic FoM shift.
contributions = (weights["R2"] * weights["sigma2"]) * baseline_a
order = np.argsort(contributions)[::-1]
selected = []
accumulated = 0.0
total = np.sum(contributions)
for index in order:
    selected.append(int(index))
    accumulated += contributions[index]
    if total > 0 and accumulated >= 0.95 * total:
        break
# Exact zeros select every column rather than interpreting a zero audit as proof.
print("Varied parameters:", names)
print("Noise columns:", [names[index] for index in selected])
print("Omitted baseline dlnFoM:", 0.5 * (total - accumulated))
for name, contribution in zip(names, contributions):
    print(name, "baseline dlnFoM contribution:", 0.5 * contribution)

## 4. Change one numerical setting at a time

A new Cobaya model is initialized for each setting, so its native grid sizes
and theory requirements are rebuilt together. The three-point sampling factor
and the ordinary two-point factor are different controls; this list states
which one changes. CAMB's integration settings and the supplied bispectrum's
sampling are not silently multiplied by a three-point control.

The diagonal addition is applied to the **same** base Fisher matrix at every
setting. Both its linearized FoM shift and exact matrix ratio remain estimates
of the noise model. They are not corrected scientific FoMs.

In [ ]:
# Each entry edits an ordinary likelihood setting from joint_shear.yaml.
# Add separate theory settings when investigating CAMB or bispectrum sampling.
candidates = {
    "default": {},
    "source_2": {"threepcf_accuracy": {"source_boost": 2.0}},
    "chi_2": {"threepcf_accuracy": {"chi_boost": 2.0}},
    "triangle_1p5": {"threepcf_accuracy": {"triangle_boost": 1.5}},
    "multipole_1p5": {"threepcf_accuracy": {"multipole_boost": 1.5}},
    "core_two_point_1p5": {"accuracyboost": 1.5},
    "sampling_2": {"threepcf_accuracy": {"accuracy_boost": 2.0}},
}
screen = {}
for label, changes in candidates.items():
    candidate_info = deepcopy(info)
    candidate_info["likelihood"]["des_3pcf.joint_shear"].update(changes)
    model = get_model(info_or_yaml_or_file=candidate_info)
    likelihood = model.likelihood["des_3pcf.joint_shear"]
    try:
        # Never carry a retained mean across changes in numerical resolution.
        predict = cnu.ExactMeanCache(predict=get_dv)
        offset_dv = cnu.make_offset_predictor(
            predict=predict,
            reference=fiducial,
            names=names,
        )
        addition = np.zeros(len(names))
        for index in selected:
            noise = cnu.get_stencil_noise(
                dv=offset_dv,
                index=index,
                h=eps[index],
                CV=np.zeros(len(names)),
                invcov=invcov,
            )
            addition[index] = noise["a"]
        estimate = cnu.predict_fom_noise(F=base, i=iw0, j=iwa, a=addition)
        screen[label] = estimate
        print(label, "estimated dlnFoM:", estimate["dlnFoM"], "ratio:", estimate["ratio"])
    finally:
        model.close()

## 5. Confirm with full derivatives and smaller steps

Choose one candidate from the screen and recalculate every column. Compare
actual marginalized FoMs at $h$ and $h/2$, as well as covariance-weighted column
changes. Repeat for another candidate if needed. Do not subtract a predicted
noise shift from the scientific FoM.

For a cusp, step halving can leave the false-information estimate nearly
constant; for a smooth fourth-order term, the fourth difference scales as
$h^4$. Shifted stencil centers, one-sided slopes and more step sizes can help
identify those behaviors. These are diagnostics of numerical derivatives,
not additional physical measurement covariance.

In [ ]:
# This cell is a real full forecast when executed. Select the setting after
# inspecting the table above; the name below is only an explicit example.
chosen = "triangle_1p5"
refined_info = deepcopy(info)
refined_info["likelihood"]["des_3pcf.joint_shear"].update(candidates[chosen])
model = get_model(info_or_yaml_or_file=refined_info)
likelihood = model.likelihood["des_3pcf.joint_shear"]
try:
    predict = cnu.ExactMeanCache(predict=get_dv)
    names, refined_center, refined_h, refined_h2 = cnu.differentiate(
        predict=predict,
        parameters=fiducial,
        steps=steps,
    )
    # The h/2 stencil uses offsets (-h,-h/2,0,h/2,h). These values
    # already live in this calculation's shared cache, including the center.
    # Retaining their coordinates makes the h/4 reuse below explicit.
    refined_stencils_h2 = np.empty((len(names), 5, refined_center.size))
    refined_coordinates_h2 = np.empty((len(names), 5))
    for column, name in enumerate(names):
        half_step = steps[name] / 2.0
        for node, offset in enumerate((-2.0, -1.0, 0.0, 1.0, 2.0)):
            point = dict(fiducial)
            if offset != 0.0:
                point[name] = float(fiducial[name]) + offset * half_step
            refined_coordinates_h2[column, node] = point[name]
            refined_stencils_h2[column, node] = predict(parameters=point)
    # These include ordinary likelihood defaults absent from a short YAML.
    # Copy the full dictionaries before changing a single angular setting.
    # The wider-step follow-up also reuses the original full-step nodes.
    # These requests are already in the cache from differentiate above.
    refined_stencils_h = np.empty_like(refined_stencils_h2)
    refined_coordinates_h = np.empty_like(refined_coordinates_h2)
    for column, name in enumerate(names):
        for node, offset in enumerate((-2, -1, 0, 1, 2)):
            point = fiducial.copy()
            if offset != 0:
                point[name] = float(fiducial[name]) + offset * steps[name]
            refined_coordinates_h[column, node] = point[name]
            refined_stencils_h[column, node] = predict(parameters=point)

    projection_settings = deepcopy(likelihood.projection_settings)
    transform_options = deepcopy(likelihood.transform_options)
    angular_accuracy = deepcopy(likelihood.threepcf_accuracy)
    if angular_accuracy is None:
        angular_accuracy = {}
    refined_fishers = []
    for derivative in (refined_h, refined_h2):
        fisher = cnu.fisher_matrix(
            derivatives=derivative,
            names=names,
            covariance=covariance,
            priors=priors,
        )
        refined_fishers.append(fisher)
        print(chosen, "actual FoM:", cnu.get_FoM(i=iw0, j=iwa, F=fisher))
    factor = np.linalg.cholesky(covariance)
    change = solve_triangular(a=factor, b=refined_h2-dh2, lower=True)
    reference = solve_triangular(a=factor, b=refined_h2, lower=True)
    for index, name in enumerate(names):
        ratio = np.linalg.norm(change[:, index]) / np.linalg.norm(reference[:, index])
        print(name, "accuracy change in derivative [%]:", 100.0 * ratio)
    directory = project / "chains/fisher_accuracy"
    directory.mkdir(parents=True, exist_ok=True)
    yaml_dump_file(str(directory / "base.yaml"), info, error_if_exists=False)
    yaml_dump_file(str(directory / "refined.yaml"), refined_info, error_if_exists=False)
    np.savez(
        file=directory / "derivatives.npz",
        names=np.asarray(names),
        steps=np.asarray(list(steps.values())),
        covariance=covariance,
        base_derivatives_h=dh,
        base_derivatives_h2=dh2,
        refined_derivatives_h=refined_h,
        refined_derivatives_h2=refined_h2,
        base_fisher=base,
        refined_fisher_h=refined_fishers[0],
        refined_fisher_h2=refined_fishers[1],
        refined_stencils_h2=refined_stencils_h2,
        refined_coordinates_h2=refined_coordinates_h2,
        row_kind=row_kind,
        prior_names=np.asarray(list(priors)),
        prior_sigmas=np.asarray(list(priors.values())),
        baseline_noise=baseline_a,
        selected_columns=np.asarray(selected),
    )
finally:
    model.close()

## 6. Compare complete Fisher information before interpreting a screen

This section uses only the derivatives already computed above. Replacing a few
columns is inexpensive, but it answers a different question from changing the
numerical settings for every parameter. Construct three derivative matrices:

* the complete default-accuracy matrix;
* the default matrix with only the selected screen columns replaced;
* the complete refined matrix.

A single-column substitution can miss a correlated change in other columns.
The complete comparison also keeps the full covariance between two- and
three-point rows. The shared routine calculates

$$F_\mathrm{joint,data}=F_{2,\mathrm{data}}+E^TS^{-1}E,$$

with $S=C_{33}-C_{32}C_{22}^{-1}C_{23}$ and
$E=D_3-C_{32}C_{22}^{-1}D_2$. The prior is then added once to each experiment.
The assertion below checks the data-only identity independently of that prior.

In [ ]:
# This is a derivative-column comparison, not another physical model run.
partial = dh2.copy()
partial[:, selected] = refined_h2[:, selected]
derivative_cases = {
    "default_h2": dh2,
    "selected_columns_h2": partial,
    "full_refined_h": refined_h,
    "full_refined_h2": refined_h2,
}
full_comparisons = {}
for label, derivatives in derivative_cases.items():
    information = cnu.compare_information(
        derivatives=derivatives,
        names=names,
        kind=row_kind,
        covariance=covariance,
        priors=priors,
    )
    np.testing.assert_allclose(
        actual=information["fisher_joint_likelihood"],
        desired=information["fisher_two_point_likelihood"]
        + information["fisher_conditional_three_point"],
        rtol=1.0e-10,
        atol=1.0e-12,
    )
    full_comparisons[label] = information
    print(label)
    for probe in ("two_point", "joint"):
        try:
            fom = cnu.figure_of_merit(
                fisher=information["fisher_" + probe],
                names=names,
                pair=("w", "wa"),
            )
        except ValueError as error:
            print(probe, "unresolved parameter combination:", error)
        else:
            print(probe, "marginalized FoM(w0,wa):", fom)

### Keep marginalization explicit when dark energy is fixed

For each calculation, first invert the full Fisher matrix to marginalize the
other parameters. To fix dark energy instead, remove the $w_0,w_a$ Fisher rows
and columns **before** inversion. Selecting a block of the already inverted
matrix would retain their marginalization.

The derived parameter $S_8=\sigma_8\sqrt{\Omega_m/0.3}$ depends on every
cosmological parameter, because $A_s$ is sampled and $\sigma_8$ is CAMB's
derived value. Its gradient $g_j=\partial S_8/\partial p_j$ comes from CAMB's
$\sigma_8$ at the same five-point stencils as the data vector, so it needs no
extra CAMB evaluation. Then $\operatorname{Var}(S_8)=g^T\operatorname{Cov}(p)g$. Retain the off-diagonal
terms. The following table compares the joint-to-two-point information gain
and IA errors for the default and fully refined calculations. A singular
parameter combination remains reported as unresolved.

In [ ]:
for label in ("default_h2", "full_refined_h2"):
    information = full_comparisons[label]
    for fixed_dark_energy in (False, True):
        retained = []
        retained_names = []
        for index, name in enumerate(names):
            if fixed_dark_energy and name in ("w", "wa"):
                continue
            retained.append(index)
            retained_names.append(name)
        amplitude_foms = {}
        for probe in ("two_point", "joint"):
            fisher = information["fisher_" + probe][np.ix_(retained, retained)]
            try:
                parameter_covariance = cnu.parameter_covariance(
                    fisher=fisher,
                    names=retained_names,
                )
            except ValueError as error:
                print(label, probe, "unresolved:", error)
                continue
            # dS8/dp of the retained parameters, from CAMB's sigma8.
            gradient = s8_gradient[retained]
            variance = gradient @ parameter_covariance @ gradient
            jacobian = np.zeros((2, len(retained)))
            jacobian[0, retained_names.index("omegam")] = 1.0
            jacobian[1] = gradient
            amplitude_covariance = jacobian @ parameter_covariance @ jacobian.T
            amplitude_foms[probe] = 1.0 / np.sqrt(np.linalg.det(amplitude_covariance))
            print(label, probe, "fixed dark energy:", fixed_dark_energy)
            print("sigma(S8):", np.sqrt(variance))
            for index, name in enumerate(retained_names):
                if name in ("DES_A1_1", "DES_A1_2", "DES_A2_1", "DES_A2_2", "DES_BTA_1"):
                    print(name, "error:", np.sqrt(parameter_covariance[index, index]))
        if "joint" in amplitude_foms and "two_point" in amplitude_foms:
            print("Joint / xi FoM(Omega_m,S8):",
                  amplitude_foms["joint"] / amplitude_foms["two_point"])

## 7. Reuse the half-step means to test a quarter step

A smaller parameter displacement does not change integration accuracy. It only
changes the finite-difference stencil. The five-point derivative with spacing
$s$ is

$$D_s=\frac{m(-2s)-8m(-s)+8m(s)-m(2s)}{12s}.$$

The refined calculation above retained its $s=h/2$ predictions at
$(-h,-h/2,0,h/2,h)$. For $s=h/4$, the two outer predictions are at
$-h/2$ and $+h/2$: these are the old stencil's second and fourth entries.
Only the predictions at $-h/4$ and $+h/4$ are new. Consequently,

$$D_{h/4}=\frac{m(-h/2)-8m(-h/4)+8m(h/4)-m(h/2)}{3h}.$$

The cell first reconstructs $D_{h/2}$ from the retained means and checks exact
agreement with the saved derivative. Then it evaluates only the two missing
points per parameter. No prediction is borrowed from another accuracy or
physical model. This is the same refined Cobaya configuration and covariance.
A fresh center evaluation checks that the reopened model reproduces the center.
Do not rerun only this section after changing the input or compiled library;
rerun the full derivative section so the retained means describe that model.

In [ ]:
model = get_model(info_or_yaml_or_file=refined_info)
likelihood = model.likelihood["des_3pcf.joint_shear"]
try:
    np.testing.assert_array_equal(
        x=ci.get_joint_covariance(),
        y=covariance,
    )
    np.testing.assert_array_equal(
        x=get_dv(parameters=fiducial),
        y=refined_center,
    )
except Exception:
    model.close()
    raise

In [ ]:
# Every row in a stencil is a full data vector. The center has zero weight
# in the first derivative but is retained to explain and check the coordinates.
quarter_derivatives = np.empty_like(refined_h2)
quarter_stencils = np.empty_like(refined_stencils_h2)
quarter_coordinates = np.empty_like(refined_coordinates_h2)
new_predictions = 0
try:
    for column, name in enumerate(names):
        step = float(steps[name])
        half = step / 2.0
        quarter = step / 4.0
        if not np.isfinite(quarter) or quarter <= 0.0:
            raise ValueError("The quarter step must be finite and positive.")
        coordinates = float(fiducial[name]) + np.array([-2, -1, 0, 1, 2]) * quarter
        if np.any(np.diff(coordinates) <= 0.0):
            raise ValueError("The quarter-step coordinates coincide in float64.")
        stored = refined_stencils_h2[column]
        expected_half_coordinates = float(fiducial[name]) + np.array([-2, -1, 0, 1, 2]) * half
        np.testing.assert_array_equal(
            x=refined_coordinates_h2[column],
            y=expected_half_coordinates,
        )
        # The stored rows sit at offsets -2, -1, 0, 1, 2; five_point takes
        # them in the order of cnu.FIVE_POINT_OFFSETS (2, 1, -1, -2), the
        # same combination cnu.differentiate used for refined_h2.
        rebuilt_half = cnu.five_point(
            vectors=[stored[4], stored[3], stored[1], stored[0]], step=half,
        )
        np.testing.assert_array_equal(x=rebuilt_half, y=refined_h2[:, column])
        np.testing.assert_array_equal(x=stored[2], y=refined_center)
        # Reuse only the outer points of the new stencil and the common center.
        quarter_stencils[column, 0] = stored[1]
        quarter_stencils[column, 2] = stored[2]
        quarter_stencils[column, 4] = stored[3]
        quarter_coordinates[column] = coordinates
        for node, offset in ((1, -1.0), (3, 1.0)):
            point = dict(fiducial)
            point[name] = float(fiducial[name]) + offset * quarter
            quarter_stencils[column, node] = get_dv(parameters=point)
            new_predictions += 1
        values = quarter_stencils[column]
        quarter_derivatives[:, column] = cnu.five_point(
            vectors=[values[4], values[3], values[1], values[0]], step=quarter,
        )
finally:
    model.close()
print("New quarter-step predictions:", new_predictions)

### Compare derivative changes in the units that affect the likelihood

Whiten a column with the Cholesky factor $C=LL^T$. Its squared length is
$D_j^TC^{-1}D_j=\|L^{-1}D_j\|^2$. The ratio of the change length to the
reference length compares derivatives without favoring the larger raw values
of the two-point rows. Multiplying by the physical step gives a predicted
mean displacement in measurement-error units.

The full FoM below is recalculated from every quarter-step column, with the
same priors. A decreasing error at $h/4$ is not assumed: numerical noise can
increase when the denominator gets smaller. These cells save new results under
`chains/fisher_accuracy`; the committed historical study arrays are unchanged.

In [ ]:
factor = np.linalg.cholesky(covariance)
whitened_reference = solve_triangular(a=factor, b=refined_h2, lower=True)
whitened_change = solve_triangular(
    a=factor,
    b=quarter_derivatives-refined_h2,
    lower=True,
)
for column, name in enumerate(names):
    reference_norm = np.linalg.norm(whitened_reference[:, column])
    difference_norm = np.linalg.norm(whitened_change[:, column])
    print(name, "step times derivative-change norm:", steps[name] * difference_norm / 4.0)
    if reference_norm == 0.0:
        print("Relative change undefined: the reference derivative is zero.")
    else:
        print("Relative derivative change [%]:", 100.0 * difference_norm / reference_norm)
quarter_information = cnu.compare_information(
    derivatives=quarter_derivatives,
    names=names,
    kind=row_kind,
    covariance=covariance,
    priors=priors,
)
for probe in ("two_point", "joint"):
    try:
        fom = cnu.figure_of_merit(
            fisher=quarter_information["fisher_" + probe],
            names=names,
            pair=("w", "wa"),
        )
    except ValueError as error:
        print(probe, "unresolved quarter-step forecast:", error)
    else:
        print(probe, "quarter-step FoM:", fom)
np.savez(
    file=directory / "quarter_step.npz",
    names=np.asarray(names),
    steps=np.asarray(list(steps.values())) / 4.0,
    mean=refined_center,
    covariance=covariance,
    derivatives_h2=refined_h2,
    derivatives_h4=quarter_derivatives,
    stencil_values_h4=quarter_stencils,
    stencil_coordinates_h4=quarter_coordinates,
    fisher_joint=quarter_information["fisher_joint"],
    fisher_two_point=quarter_information["fisher_two_point"],
)

## Separate step size from numerical resolution

A smaller finite-difference step need not be better: it reduces smooth
truncation error but magnifies changes caused by interpolation and integration.
We therefore also test twice the original spacing for matter density and
Hubble $h$. Here $\Delta_j$ denotes a parameter step; Hubble $h$ is a parameter.
The wider stencil is

$$
D_j(2\Delta_j)=\frac{m(p_j-4\Delta_j)-8m(p_j-2\Delta_j)
 +8m(p_j+2\Delta_j)-m(p_j+4\Delta_j)}{24\Delta_j}.
$$
The stored original stencil already supplies the two inner values. Only
$\pm4\Delta_j$ are new evaluations. The following model creation and loop are
expensive; run them when the earlier full refinement is available. The final
column substitutions hold all other derivatives at the refined half-step.
They diagnose these two columns; they are not complete wider-step forecasts.

In [ ]:
model = get_model(info_or_yaml_or_file=deepcopy(refined_info))
likelihood = next(iter(model.likelihood.values()))
try:
    np.testing.assert_array_equal(ci.get_joint_covariance(), covariance)
    np.testing.assert_array_equal(get_dv(parameters=fiducial), refined_center)
except Exception:
    model.close()
    raise

In [ ]:
wide_derivatives = refined_h2.copy()
wide_stencils = {}
wide_predictions = 0
try:
    for name in ("omegam", "H0"):
        column = names.index(name)
        step = float(steps[name])
        coordinates = float(fiducial[name]) + np.array([-4, -2, 0, 2, 4]) * step
        if not np.all(np.isfinite(coordinates)) or np.any(np.diff(coordinates) <= 0):
            raise ValueError("The wider-step coordinates must be distinct and finite.")
        stored = refined_stencils_h[column]
        expected = float(fiducial[name]) + np.array([-2, -1, 0, 1, 2]) * step
        np.testing.assert_array_equal(refined_coordinates_h[column], expected)
        # Rows at offsets -2, -1, 0, 1, 2, taken in the order of
        # cnu.FIVE_POINT_OFFSETS (2, 1, -1, -2), as cnu.differentiate does.
        rebuilt = cnu.five_point(
            vectors=[stored[4], stored[3], stored[1], stored[0]], step=step,
        )
        np.testing.assert_array_equal(rebuilt, refined_h[:, column])
        np.testing.assert_array_equal(stored[2], refined_center)
        values = np.empty_like(stored)
        # Original outer nodes become inner nodes at twice the spacing.
        values[1] = stored[0]
        values[2] = stored[2]
        values[3] = stored[4]
        for node, offset in ((0, -4), (4, 4)):
            point = fiducial.copy()
            point[name] = float(fiducial[name]) + offset * step
            values[node] = get_dv(parameters=point)
            wide_predictions += 1
        wide_derivatives[:, column] = cnu.five_point(
            vectors=[values[4], values[3], values[1], values[0]], step=2.0 * step,
        )
        wide_stencils[name] = values.copy()
finally:
    model.close()
print("New wider-step predictions:", wide_predictions)

In [ ]:
cosmology_columns = [names.index(name) for name in ("omegam", "H0")]
step_derivatives = {
    "quarter": quarter_derivatives,
    "half": refined_h2,
    "original": refined_h,
    "double": wide_derivatives,
}
for label, derivatives in step_derivatives.items():
    partial = refined_h2.copy()
    partial[:, cosmology_columns] = derivatives[:, cosmology_columns]
    fisher = cnu.fisher_matrix(
        derivatives=partial,
        covariance=covariance,
        names=names,
        priors=priors,
    )
    merit = cnu.figure_of_merit(fisher=fisher, names=names, pair=["w", "wa"])
    print(label, "selected-column dark-energy FoM", merit)
np.savez(
    file=directory / "wider_step.npz",
    names=np.asarray(names),
    derivatives=wide_derivatives,
    omegam=wide_stencils["omegam"],
    H0=wide_stencils["H0"],
)

## Why small derivative changes can move the marginalized ellipse

Whitening turns the covariance metric into an ordinary Euclidean dot product.
Write $C=LL^{\mathsf T}$ and whiten every derivative with $L^{-1}$. To include
independent Gaussian priors once, append one row per constrained parameter,
with entry $1/\sigma_j$ in its column and zero elsewhere. Call this enlarged
matrix $Z$. Then

$$F=Z^{\mathsf T}Z.$$

Separate its columns into the target pair $Z_w$ and all nuisance columns
$Z_n$. A QR decomposition supplies an orthonormal basis $Q_n$ for the nuisance
span. Subtracting the component along that span leaves

$$E_w=Z_w-Q_n(Q_n^{\mathsf T}Z_w).$$

The surviving information is $G=E_w^{\mathsf T}E_w$, and the marginalized
pair FoM is $\sqrt{\det G}$. A small fraction of a full response can survive
this projection. A change that is small before projection can therefore be
large compared with the surviving response. This is a numerical diagnostic,
not evidence for additional physical information.

In [ ]:
# Priors are appended only here; derivatives contain likelihood responses only.
prior_rows = np.zeros((len(names), len(names)))
for name, sigma in priors.items():
    prior_rows[names.index(name), names.index(name)] = 1.0 / sigma
factor = np.linalg.cholesky(covariance)
target = [names.index("w"), names.index("wa")]
nuisance = [index for index in range(len(names)) if index not in target]
augmented = {}
effective_information = {}
target_residuals = {}
projection_metrics = {}
for label, derivatives in {
    "default_half": dh2,
    "refined_half": refined_h2,
    "refined_original": refined_h,
}.items():
    whitened = solve_triangular(a=factor, b=derivatives, lower=True)
    response = np.vstack((whitened, prior_rows))
    augmented[label] = response
    column_norm = np.linalg.norm(response, axis=0)
    if np.any(column_norm == 0):
        raise ValueError("A zero response column needs an explicit parameter decision.")
    scaled = response / column_norm[None, :]
    singular = np.linalg.svd(scaled, compute_uv=False)
    # Do not replace an unresolved inverse by a pseudoinverse or an eigenvalue floor.
    fisher = response.T @ response
    parameter_covariance = cnu.parameter_covariance(fisher=fisher, names=names)
    basis, triangular = np.linalg.qr(scaled[:, nuisance], mode="reduced")
    if np.linalg.matrix_rank(triangular) != len(nuisance):
        raise ValueError("The nuisance response span is unresolved.")
    residual = response[:, target] - basis @ (basis.T @ response[:, target])
    effective = residual.T @ residual
    effective_information[label] = effective
    target_residuals[label] = residual
    eigenvalues, eigenvectors = np.linalg.eigh(scaled.T @ scaled)
    target_values, target_vectors = np.linalg.eigh(effective)
    pair_covariance = parameter_covariance[np.ix_(target, target)]
    merit_qr = np.sqrt(np.linalg.det(effective))
    merit_inverse = cnu.figure_of_merit(fisher=fisher, names=names, pair=["w", "wa"])
    # An SVD of Z checks the ellipse without first forming the Fisher Gram matrix.
    _, values, transpose = np.linalg.svd(scaled, full_matrices=False)
    covariance_svd = (
        (transpose.T / values**2) @ transpose
    ) / column_norm[:, None] / column_norm[None, :]
    merit_svd = 1 / np.sqrt(np.linalg.det(covariance_svd[np.ix_(target, target)]))
    largest = np.argsort(np.abs(eigenvectors[:, 0]))[::-1][:8]
    projection_metrics[label] = {
        "fom_qr": merit_qr,
        "fom_inverse": merit_inverse,
        "fom_svd": merit_svd,
        "qr_inverse_relative_difference": merit_qr / merit_inverse - 1,
        "normalized_eigenvalues": eigenvalues,
        "normalized_condition": eigenvalues[-1] / eigenvalues[0],
        "singular_ratio": singular[-1] / singular[0],
        "weak_normalized_direction": dict(zip(
            np.asarray(names)[largest],
            eigenvectors[largest, 0],
        )),
        "target_information_eigenvalues": target_values,
        "weak_target_unit_direction": target_vectors[:, 0],
        "target_sigma": np.sqrt(np.diag(pair_covariance)),
        "target_correlation": pair_covariance[0, 1] / np.sqrt(np.prod(np.diag(pair_covariance))),
        "surviving_response_fraction": np.linalg.norm(residual, axis=0) / column_norm[target],
    }
    print(label, projection_metrics[label])

Compare the default and refined information along the same physical directions.
If $G_0=KK^{\mathsf T}$, the eigenvalues of $K^{-1}G_1K^{-\mathsf T}$ measure
the information ratios along its two eigenvectors. Transforming those vectors
back with $K^{-\mathsf T}$ recovers directions in $(w_0,w_a)$ coordinates.
Unlike the determinant alone, these two ratios expose unequal changes along
the ellipse axes.

To examine the baseline's weakest target direction $v$, choose nuisance
shifts $q$ that minimize $\|Z_wv+Z_nq\|$. Solve the triangular system from QR;
then compare the response to this *same* displacement before and after the
grid change. The cosine retains its sign: opposite changes can cancel, while
aligned changes reinforce one another.

In [ ]:
current = augmented["default_half"]
updated = augmented["refined_half"]
effective0 = effective_information["default_half"]
effective1 = effective_information["refined_half"]
target_factor = np.linalg.cholesky(effective0)
relative = solve_triangular(a=target_factor, b=effective1, lower=True)
relative = solve_triangular(a=target_factor, b=relative.T, lower=True).T
ratios, vectors = np.linalg.eigh(0.5 * (relative + relative.T))
directions = solve_triangular(a=target_factor.T, b=vectors, lower=False)
print("Generalized information ratios:", ratios)
print("Corresponding w0, wa directions (columns):", directions)
_, vectors = np.linalg.eigh(effective0)
direction = vectors[:, 0]
basis, triangular = np.linalg.qr(current[:, nuisance], mode="reduced")
coefficients = np.zeros(len(names))
coefficients[target] = direction
coefficients[nuisance] = solve_triangular(
    a=triangular,
    b=-basis.T @ (current[:, target] @ direction),
    lower=False,
)
response = current @ coefficients
change = (updated - current) @ coefficients
response_norm = np.linalg.norm(response)
change_norm = np.linalg.norm(change)
print("Weak displacement:", dict(zip(names, coefficients)))
print("Baseline response norm:", response_norm)
print("Change norm and fractional change:", change_norm, change_norm / response_norm)
if change_norm > 0:
    print("Signed response/change cosine:", response @ change / (response_norm * change_norm))
for position, column in enumerate(target):
    old = target_residuals["default_half"][:, position]
    new = target_residuals["refined_half"][:, position]
    relative_change = np.linalg.norm(new - old) / np.linalg.norm(old)
    print(names[column], "projected relative change", relative_change)
# Compare grid changes with step changes without asserting either is the exact answer.
step_change = augmented["refined_original"] - updated
grid_change = updated - current
for column, name in enumerate(names):
    denominator = np.linalg.norm(step_change[:, column]) * np.linalg.norm(grid_change[:, column])
    cosine = np.nan
    if denominator > 0:
        cosine = step_change[:, column] @ grid_change[:, column] / denominator
    relative_norm = np.linalg.norm(grid_change[:, column]) / np.linalg.norm(current[:, column])
    print(name, "relative grid change", relative_norm, "grid/step cosine", cosine)

## Identify which columns and observables drive a change

The next substitutions replace either two-point rows, aperture rows, or both
for selected parameter columns. They always rebuild the information with the
*full* covariance, including cross-covariance, and the same priors. A hybrid
matrix is useful for locating sensitivity; it is not a forecast at a uniform
accuracy setting. Comparing the two row blocks separately must not replace
their joint covariance by independent blocks.

In [ ]:
# Separate sensitivity to a grid change from sensitivity to derivative spacing.
# Each reference uses the same physical fiducial, covariance, parameter order and priors.
block_substitutions = {}
for experiment, fine, coarse in (
    ("grid", dh2, refined_h2),
    ("step", refined_h2, refined_h),
):
    reference_fisher = cnu.fisher_matrix(
        derivatives=fine,
        covariance=covariance,
        names=names,
        priors=priors,
    )
    reference_merit = cnu.figure_of_merit(
        fisher=reference_fisher,
        names=names,
        pair=["w", "wa"],
    )
    for members in (("w", "wa"), ("omegam",), ("H0",), ("omegam", "H0")):
        columns = [names.index(name) for name in members]
        for label, rows in (
            ("xi", np.flatnonzero(row_kind != "map3")),
            ("Map3", np.flatnonzero(row_kind == "map3")),
            ("joint", np.arange(fine.shape[0])),
        ):
            derivative = fine.copy()
            derivative[np.ix_(rows, columns)] = coarse[np.ix_(rows, columns)]
            fisher = cnu.fisher_matrix(
                derivatives=derivative,
                covariance=covariance,
                names=names,
                priors=priors,
            )
            ratio = cnu.figure_of_merit(
                fisher=fisher,
                names=names,
                pair=["w", "wa"],
            ) / reference_merit
            key = experiment + ":" + ",".join(members) + ":" + label
            block_substitutions[key] = ratio
            print(key, "FoM ratio", ratio)
print("Maximum raw xi derivative change:", np.max(np.abs((refined_h2 - dh2)[row_kind != "map3"])))
# Finally replace one complete column at a time. This includes nuisance columns
# that can rotate the marginalized ellipse even when their own errors look stable.
for column, name in enumerate(names):
    partial = refined_h2.copy()
    partial[:, column] = refined_h[:, column]
    fisher = cnu.fisher_matrix(
        derivatives=partial,
        covariance=covariance,
        names=names,
        priors=priors,
    )
    ratio = cnu.figure_of_merit(fisher=fisher, names=names, pair=["w", "wa"]) / reference_merit
    block_substitutions["single_column:" + name] = ratio
    print(name, "single-column step FoM ratio", ratio)


## 8. Test angular truncation separately from interpolation accuracy

The projection's maximum Legendre degree $L$ and the transform's maximum Fourier
order $M$ truncate angular expansions. Increasing a table's sampling boost does
not raise these truncation orders. The cases below first raise $L$, then raise
$M$, then separately refine the angular quadrature and replace the geometric
coupling interpolation with its finite analytic sum. Finally they combine the
last two changes. They retain the selected refined grid and every physical
parameter; this is a **mean-only** check, not a new Fisher convergence result.

For each case, compare the mean with the refined baseline using the full joint
covariance. The two-point calculation has no dependence on these three-point
angular settings, so its entries should remain unchanged. The changes in the
joint distance can expose a missing angular mode even when a simple fractional
comparison is unstable near a zero crossing.

In [ ]:
# Tuple columns: Legendre cutoff, Fourier cutoff, doubled angular quadrature,
# and exact geometric coupling. The setting npsi=0 requests that exact sum.
angular_cases = {
    "L50_M20": (50, 20, False, False),
    "L50_M50": (50, 50, False, False),
    "L50_M50_angular": (50, 50, True, False),
    "L50_M50_coupling": (50, 50, False, True),
    "L50_M50_both": (50, 50, True, True),
}
angular_results = {}
for label, (lmax, mmax, double_angular, exact_coupling) in angular_cases.items():
    angular_info = deepcopy(refined_info)
    settings = angular_info["likelihood"]["des_3pcf.joint_shear"]
    settings["map_lmax"] = lmax
    settings["map_mmax"] = mmax
    settings["projection_settings"] = deepcopy(projection_settings)
    settings["projection_settings"]["lmax"] = lmax
    settings["transform_options"] = deepcopy(transform_options)
    settings["threepcf_accuracy"] = deepcopy(angular_accuracy)
    if double_angular:
        settings["threepcf_accuracy"]["angular_boost"] = (
            2.0 * angular_accuracy.get("angular_boost", 1.0)
        )
    if exact_coupling:
        settings["transform_options"]["npsi"] = 0
    model = get_model(info_or_yaml_or_file=angular_info)
    likelihood = model.likelihood["des_3pcf.joint_shear"]
    try:
        angular_mean = get_dv(parameters=fiducial)
        np.testing.assert_array_equal(
            x=angular_mean[row_kind != "map3"],
            y=refined_center[row_kind != "map3"],
        )
        difference = angular_mean - refined_center
        whitened = solve_triangular(a=factor, b=difference, lower=True)
        chi2_difference = whitened @ whitened
        angular_results[label] = chi2_difference
        print(label, "mean difference chi2:", chi2_difference)
        yaml_dump_file(
            file_name=str(directory / (label + ".yaml")),
            data=angular_info,
            error_if_exists=False,
        )
        np.savez(
            file=directory / (label + ".npz"),
            mean=angular_mean,
            reference_mean=refined_center,
            covariance=covariance,
            delta_chi2=chi2_difference,
        )
    finally:
        model.close()